In [1]:
import pandas as pd
import xmltodict

In [ ]:
with open('data/ecgen-radiology/1.xml', 'rb') as fp:
	df = xmltodict.parse(fp)

FileNotFoundError: [Errno 2] No such file or directory: 'ecgen-radiology/1.xml'

In [4]:
def get_row(abstract):
	row = ""

	for x in abstract:
		if x['@Label'] == 'INDICATION':
			row += f"{x.get("#text", "")}; "
		elif x['@Label'] == 'FINDINGS':
			row += f"{x.get("#text", "")}\n"
	return row

In [6]:
with open('processed_data.csv', 'w') as fp:
	fp.write('INDICATION; FINDINGS\n')

for i in range(1, 10):
	with open(f'data/ecgen-radiology/{i}.xml', 'rb') as fp:
		xml_dict = xmltodict.parse(fp)
	abstract = xml_dict['eCitation']['MedlineCitation']['Article']['Abstract']['AbstractText']
	print(abstract)
	row = get_row(abstract)
	with open('processed_data.csv', 'a') as fp:
		fp.write(row)

[{'@Label': 'COMPARISON', '#text': 'None.'}, {'@Label': 'INDICATION', '#text': 'Positive TB test'}, {'@Label': 'FINDINGS', '#text': 'The cardiac silhouette and mediastinum size are within normal limits. There is no pulmonary edema. There is no focal consolidation. There are no XXXX of a pleural effusion. There is no evidence of pneumothorax.'}, {'@Label': 'IMPRESSION', '#text': 'Normal chest x-XXXX.'}]
[{'@Label': 'COMPARISON', '#text': 'None.'}, {'@Label': 'INDICATION', '#text': 'Preop bariatric surgery.'}, {'@Label': 'FINDINGS', '#text': 'Borderline cardiomegaly. Midline sternotomy XXXX. Enlarged pulmonary arteries. Clear lungs. Inferior XXXX XXXX XXXX.'}, {'@Label': 'IMPRESSION', '#text': 'No acute pulmonary findings.'}]
[{'@Label': 'COMPARISON'}, {'@Label': 'INDICATION', '#text': 'rib pain after a XXXX, XXXX XXXX steps this XXXX. Pain to R back, R elbow and R rib XXXX, no previous heart or lung hx, non-XXXX, no hx ca'}, {'@Label': 'FINDINGS'}, {'@Label': 'IMPRESSION', '#text': 'No 

In [70]:
df = pd.read_csv('data/processed_data.csv', sep='$')

df = df.dropna()

In [71]:
df.head(10)

,IMPRESSION,FINDINGS
1,1. Focal opacity in the right midlung zone wor...,Cardiomediastinal silhouette is unchanged with...
2,Surgical changes of the right hemithorax and m...,Lungs are clear bilaterally with no focal infi...
3,No acute cardiopulmonary abnormality. .,Heart size within normal limits. No focal airs...
4,No acute cardiopulmonary findings.,No focal lung consolidation. Heart size and pu...
5,No acute cardiopulmonary process.,Redemonstration of colonic interposition overl...
6,1. Bibasilar airspace disease left greater tha...,Lungs are clear bilaterally.There is no focal ...
7,No acute cardiopulmonary abnormality.,The lungs are clear. Heart size is normal. No ...
10,"Comparison XXXX, XXXX. Well-expanded and clear...",Normal heart size and mediastinal contours. Ca...
11,No acute cardiopulmonary abnormality.,The lungs are clear. There is no pleural effus...
13,Small calcification right lung base with appea...,Heart size within normal limits. Streaky airsp...


In [13]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorForLanguageModeling

model_name = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
dataset = load_dataset("karthiksagarn/astro_horoscope", split="train")

dataset = dataset.select(range(1000))

def tokenize(batch):
    return tokenizer(
        batch["horoscope"],
        truncation=True,
        max_length=512,
    )

dataset = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)
dataset = dataset.train_test_split(test_size=0.1)

Map: 100%|██████████| 1000/1000 [00:00<00:00, 13476.89 examples/s]


In [14]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model_name = "Qwen/Qwen3-0.6B"
model = AutoModelForCausalLM.from_pretrained(model_name, dtype="auto")

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 1601.43it/s]


In [ ]:
training_args = TrainingArguments(
    output_dir="qwen3-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    bf16=True,
    learning_rate=2e-5,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss
1,3.594386,3.621137
2,3.576013,3.621130
3,3.563208,3.621130


Writing model shards: 100%|██████████| 1/1 [00:06<00:00,  6.87s/it]
There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=171, training_loss=3.552960571489836, metrics={'train_runtime': 671.4856, 'train_samples_per_second': 4.021, 'train_steps_per_second': 0.255, 'total_flos': 548503551737856.0, 'train_loss': 3.552960571489836, 'epoch': 3.0})

In [63]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from dotenv import load_dotenv
import os

load_dotenv()

model_string = "Qwen/Qwen3-0.6B"

with open("prompts/prompt_template.txt") as f:
	prompt = f.read().format(df.iloc[0]["FINDINGS"])

from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(model_string)
model = AutoModelForCausalLM.from_pretrained(model_string)
messages = [
    {"role": "user", "content": prompt},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=1000)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

<think>
Okay, let's see. The user provided some findings from a medical context and wants an impression. First, I need to parse through the findings to understand what's being discussed.

The first point mentions the cardiomediastinal silhouette remains unchanged with mild cardiomegaly. That's important for understanding the chest structure. Then there's a relative elevation of the right hemidiaphragm, which might relate to a history of right lower lobectomy. They also note that there's no focal consolidation, pneumothorax, or effusion. The irregularity of the right fifth and sixth ribs is mentioned since at some points after surgery. The left shoulder rotator bone anchor is present, and no acute osseous abnormalities.

Now, the user wants an impression. The impression should summarize these findings in a concise manner. I need to make sure not to miss any key points. Let me check each finding again to confirm. The silhouette is unchanged, right hemidiaphragm elevation is consistent wi

In [73]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from dotenv import load_dotenv
import os

load_dotenv()

model_string = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_string)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_string,
    dtype=torch.bfloat16,
    device_map="auto",
	token=os.getenv("HF_TOKEN")
)

with open("prompts/prompt_template.txt") as f:
	prompt = f.read().format(df.iloc[0]["FINDINGS"])

input_ids = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(**input_ids, max_new_tokens=100)
print(prompt)
print(tokenizer.decode(output[0], skip_special_tokens=True))
print(df.iloc[0]["IMPRESSION"])

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Findings: Cardiomediastinal silhouette is unchanged with mild cardiomegaly. There is relative elevation of the right hemidiaphragm consistent with history of right lower lobectomy, without focal consolidation, pneumothorax, or effusion identified. Irregularity of the right fifth and sixth ribs stable since at XXXX XXXX and XXXX postsurgical/post traumatic in XXXX. Left shoulder rotator XXXX bone anchor noted. No acute osseous abnormality identified.
Impression:

The right hemidiaphragm is stable with no vascular abnormalities.
1. Focal opacity in the right midlung zone worrisome for pneumonitis. 2. Mild pulmonary vascular congestion.


In [76]:
def tokenize_function(example):
    prompt = [f"Findings: {example["FINDINGS"]}\nImpression:" for findings in example["FINDINGS"]]
    example['input_ids'] = tokenizer(prompt, padding="max_length", truncation=True, return_tensors="pt").input_ids
    example['labels'] = tokenizer(example["IMPRESSION"], padding="max_length", truncation=True, return_tensors="pt").input_ids
    
    return example

In [84]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(0.1)
dataset_test = dataset["test"].train_test_split(0.5)
dataset["test"] = dataset_test["train"]
dataset["validation"] = dataset_test["test"]

tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(['IMPRESSION', 'FINDINGS', '__index_level_0__'])

Map:   0%|          | 0/3051 [00:00<?, ? examples/s]

Map:   0%|          | 0/170 [00:00<?, ? examples/s]

Map:   0%|          | 0/170 [00:00<?, ? examples/s]

In [85]:
print(f"Shapes of the datasets:")
print(f"Training: {tokenized_datasets['train'].shape}")
print(f"Validation: {tokenized_datasets['validation'].shape}")
print(f"Test: {tokenized_datasets['test'].shape}")
print(tokenized_datasets)

Shapes of the datasets:
Training: (3051, 2)
Validation: (170, 2)
Test: (170, 2)
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 3051
    })
    test: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 170
    })
    validation: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 170
    })
})


In [86]:
import time
from transformers import TrainingArguments, Trainer
output_dir = f'./checkpoints/flan-t5-finetuned/model-{str(int(time.time()))}'

training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=1e-5,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=1,
    max_steps=1
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation']
)

trainer.train()

Step,Training Loss
1,39.250000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1, training_loss=39.25, metrics={'train_runtime': 57.1367, 'train_samples_per_second': 0.14, 'train_steps_per_second': 0.018, 'total_flos': 5478058819584.0, 'train_loss': 39.25, 'epoch': 0.002617801047120419})

In [87]:
# Save the trained model
trained_model_dir = f'./models/flan-t5-finetuned/trained-model-1'
trainer.save_model(trained_model_dir)

# Load the trained model
trained_model = AutoModelForSeq2SeqLM.from_pretrained("./models/flan-t5-finetuned/trained-model-1")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [92]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from dotenv import load_dotenv
import os

load_dotenv()

model_string = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_string)
trained_model = AutoModelForSeq2SeqLM.from_pretrained("./models/flan-t5-finetuned/trained-model-1")

with open("prompts/prompt_template.txt") as f:
	prompt = f.read().format(dataset["test"]["FINDINGS"][5])
print(prompt)
print(dataset["test"]["IMPRESSION"][5])
input_ids = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(**input_ids, max_new_tokens=1000)
print(tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Findings: The lungs and pleural spaces show no acute abnormality. Heart size upper limits of normal, predominantly left ventricular contour (XXXX visualized on lateral projection), pulmonary vascularity within normal limits. .
Impression:
No comparison chest x-XXXX. Well-expanded and clear lungs. Mediastinal contour within normal limits. No acute cardiopulmonary abnormality identified.
No pulmonary vascular disease.


In [ ]:
findings = dataset["test"]["FINDINGS"]
original_impressions = dataset["test"]["IMPRESSION"]
model_impressions = []

for x in findings:
	prompt = f"Findings: {x}\nImpression: "
	input_ids = tokenizer(prompt, return_tensors="pt").to(model.device)
	output = model.generate(**input_ids, max_new_tokens=1000)
	text = tokenizer.decode(output[0], skip_special_tokens=True)
	model_impressions.append(text)

from rogue import Rogue

Column(['Lungs are clear. No pleural effusions or pneumothoraces. Heart and mediastinum of normal size and contour. scoliosis.', 'The cardiomediastinal silhouette and pulmonary vasculature are within normal limits in size. There is a thin right apical pneumothorax measuring approximately 5 mm in thickness. There is extensive subcutaneous emphysema in the right chest wall and neck. There are fractures of the right anterior 5th through 9th anterior ribs with mild displacement. Additional fractures cannot entirely be excluded. There is mild streaky airspace disease in the right lung base. Left lung is clear. There is a small hiatal hernia. There is an intrathecal catheter terminating in the lower thoracic spine.', 'Both lungs remain clear and expanded. Heart and pulmonary XXXX are normal. No change in the large hiatus hernia.', 'The heart size and mediastinal contours appear within normal limits. Atherosclerotic calcification of the aorta. No focal airspace consolidation, pleural effusion

In [ ]:
from transformers import pipeline

with open("prompts/system-prompt.txt") as f:
	system_prompt = f.read()

with open("prompts/prompt_template.txt") as f:
	prompt = f.read().format(df.iloc[0]["FINDINGS"])

pipe = pipeline("text-generation", model="Qwen/Qwen3.5-0.8B")


messages = [
    {
		"role": "system",
		"content": [
			{"type": "text", "text": system_prompt}
        ],
        "role": "user",
        "content": [
            {"type": "text", "text": prompt}
        ]
    },
]

pipe(text_inputs=messages)

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': [{'role': 'user',
    'content': [{'type': 'text',
      'text': 'Findings: There is a focal area of opacity in the right midlung zone. This was not present on the recent prior study. There is prominence of the pulmonary markings throughout and there are small bilateral pleural effusions. The heart is not significantly enlarged. There is a prosthetic valve. There are atherosclerotic changes of the aorta. Arthritic changes of the skeletal structures are noted.\nIndication:\n'}]},
   {'role': 'assistant',
    'content': 'Based on the findings provided in the clinical vignette, the patient is presenting with several signs and symptoms consistent with a primary pulmonary pathology affecting the lung parenchyma.\n\nThe key features include:\n*   **Focal opacity in the right midlung zone:** This suggests a consolidation, ground-glass opacity, or mass, which are classic radiological signs of **Pulmonary Fibrosis**.\n*   **Prominent pulmonary markings:** This indicates hype